In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv(
    r"C:\Users\Usuario\Documents\mdc\TradingBacktest\datos\ES_1min.txt",
    sep=";",
    header=None,
    names=["datetime", "open", "high", "low", "close", "volume"]
)

df["datetime"] = pd.to_datetime(df["datetime"], format="%Y%m%d %H%M%S")
df = df.set_index("datetime")

print(f"Total de barras: {len(df)}")
print(f"Desde: {df.index[0]}")
print(f"Hasta: {df.index[-1]}")
print(df.head())

Total de barras: 35805
Desde: 2026-03-12 04:01:00
Hasta: 2026-04-17 12:55:00
                        open     high      low    close  volume
datetime                                                       
2026-03-12 04:01:00  6762.50  6762.50  6762.50  6762.50       1
2026-03-12 04:03:00  6764.00  6764.25  6763.75  6764.25       9
2026-03-12 04:05:00  6764.75  6766.00  6764.75  6765.75      24
2026-03-12 04:06:00  6766.00  6767.00  6766.00  6766.25      40
2026-03-12 04:07:00  6765.75  6766.00  6765.75  6766.00       3


In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv(
    r"C:\Users\Usuario\Documents\mdc\TradingBacktest\datos\ES_1min.txt",
    sep=";",
    header=None,
    names=["datetime", "open", "high", "low", "close", "volume"]
)

df["datetime"] = pd.to_datetime(df["datetime"], format="%Y%m%d %H%M%S")
df = df.set_index("datetime")

print(f"Total de barras: {len(df)}")
print(f"Desde: {df.index[0]}")
print(f"Hasta: {df.index[-1]}")
print(df.head())

Total de barras: 35805
Desde: 2026-03-12 04:01:00
Hasta: 2026-04-17 12:55:00
                        open     high      low    close  volume
datetime                                                       
2026-03-12 04:01:00  6762.50  6762.50  6762.50  6762.50       1
2026-03-12 04:03:00  6764.00  6764.25  6763.75  6764.25       9
2026-03-12 04:05:00  6764.75  6766.00  6764.75  6765.75      24
2026-03-12 04:06:00  6766.00  6767.00  6766.00  6766.25      40
2026-03-12 04:07:00  6765.75  6766.00  6765.75  6766.00       3


In [5]:
import pytz

eastern = pytz.timezone("America/New_York")
df.index = df.index.tz_localize("America/New_York")

# Filtrar solo el período SHORT y horario de sesión regular
start = "2026-03-12"
end = "2026-04-01"
df_period = df[start:end]

# Filtrar horario 9:30 a 16:00 ET
df_session = df_period.between_time("09:30", "16:00")

# Construir velas de 5 minutos
df_5m = df_session.resample("5min").agg({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum"
}).dropna()

print(f"Total barras de 5 minutos: {len(df_5m)}")
print(f"Desde: {df_5m.index[0]}")
print(f"Hasta: {df_5m.index[-1]}")
print(df_5m.head(10))

Total barras de 5 minutos: 1185
Desde: 2026-03-12 09:30:00-04:00
Hasta: 2026-04-01 16:00:00-04:00
                              open     high      low    close  volume
datetime                                                             
2026-03-12 09:30:00-04:00  6792.50  6797.00  6791.50  6796.50     117
2026-03-12 09:35:00-04:00  6795.75  6799.00  6795.00  6798.00      71
2026-03-12 09:40:00-04:00  6799.00  6805.00  6798.50  6804.00     217
2026-03-12 09:45:00-04:00  6804.25  6807.00  6803.00  6805.75     118
2026-03-12 09:50:00-04:00  6805.50  6808.50  6803.50  6808.50      83
2026-03-12 09:55:00-04:00  6808.75  6809.00  6806.50  6807.25      82
2026-03-12 10:00:00-04:00  6807.00  6810.25  6806.50  6807.50     124
2026-03-12 10:05:00-04:00  6807.75  6808.50  6804.50  6804.75      89
2026-03-12 10:10:00-04:00  6804.25  6804.75  6802.50  6803.25      36
2026-03-12 10:15:00-04:00  6803.75  6806.25  6801.75  6803.25      64


In [6]:
trades = []
dias = df_5m.groupby(df_5m.index.date)

for fecha, dia in dias:
    barras = dia.copy()
    if len(barras) < 6:
        continue
    
    # Rango ORB primeras 3 velas (9:30, 9:35, 9:40)
    orb = barras.iloc[:3]
    orb_high = orb["high"].max()
    orb_low = orb["low"].min()
    orb_rango = orb_high - orb_low
    
    # Filtro volatilidad
    if orb_rango > 20:
        continue
    
    # Buscar entrada SHORT desde vela 4 en adelante
    entrada = None
    for i in range(3, len(barras)):
        vela = barras.iloc[i]
        if vela["close"] < orb_low:
            entrada = {
                "fecha": fecha,
                "hora_entrada": barras.index[i],
                "precio_entrada": vela["close"],
                "orb_high": orb_high,
                "orb_low": orb_low,
                "orb_rango": orb_rango,
                "stop": orb_high,
                "target": vela["close"] - (orb_rango * 1.5)
            }
            break
    
    if entrada is None:
        continue
    
    # Simular resultado
    resultado = None
    for i in range(barras.index.get_loc(entrada["hora_entrada"]) + 1, len(barras)):
        vela = barras.iloc[i]
        hora = barras.index[i].time()
        
        # Cierre forzado 15:45
        if hora >= pd.Timestamp("15:45").time():
            entrada["salida"] = vela["close"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "tiempo"
            entrada["pnl"] = entrada["precio_entrada"] - vela["close"]
            resultado = entrada
            break
        
        # Stop hit
        if vela["high"] >= entrada["stop"]:
            entrada["salida"] = entrada["stop"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "stop"
            entrada["pnl"] = entrada["precio_entrada"] - entrada["stop"]
            resultado = entrada
            break
        
        # Target hit
        if vela["low"] <= entrada["target"]:
            entrada["salida"] = entrada["target"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "target"
            entrada["pnl"] = entrada["precio_entrada"] - entrada["target"]
            resultado = entrada
            break
    
    if resultado:
        trades.append(resultado)

df_trades = pd.DataFrame(trades)
print(f"Total trades: {len(df_trades)}")
print(df_trades[["fecha","precio_entrada","stop","target","resultado","pnl"]].to_string())

Total trades: 13
         fecha  precio_entrada     stop    target resultado     pnl
0   2026-03-12         6788.75  6805.00  6768.500    target  20.250
1   2026-03-13         6716.25  6726.25  6708.375      stop -10.000
2   2026-03-17         6735.50  6748.00  6725.375      stop -12.500
3   2026-03-18         6806.25  6810.00  6801.375      stop  -3.750
4   2026-03-19         6645.50  6663.25  6619.625    target  25.875
5   2026-03-20         6638.00  6654.50  6618.125    target  19.875
6   2026-03-23         6498.25  6518.75  6468.250      stop -20.500
7   2026-03-24         6607.25  6626.25  6590.000      stop -19.000
8   2026-03-25         6651.75  6662.50  6642.375      stop -10.750
9   2026-03-26         6577.00  6591.75  6561.250      stop -14.750
10  2026-03-27         6518.25  6532.00  6504.000    target  14.250
11  2026-03-30         6430.50  6440.50  6417.750    target  12.750
12  2026-03-31         6427.75  6446.25  6414.250      stop -18.500


In [7]:
# Valor del punto ES = $50 por punto
punto = 50

df_trades["pnl_usd"] = df_trades["pnl"] * punto
df_trades["equity"] = df_trades["pnl_usd"].cumsum()

# Estadísticas
total = len(df_trades)
ganadores = df_trades[df_trades["pnl"] > 0]
perdedores = df_trades[df_trades["pnl"] < 0]
win_rate = len(ganadores) / total * 100
pnl_total = df_trades["pnl_usd"].sum()
ganancia_media = ganadores["pnl_usd"].mean()
perdida_media = perdedores["pnl_usd"].mean()
profit_factor = ganadores["pnl_usd"].sum() / abs(perdedores["pnl_usd"].sum())
max_drawdown = (df_trades["equity"].cummax() - df_trades["equity"]).max()

por_resultado = df_trades["resultado"].value_counts()

print("=" * 40)
print("ESTADÍSTICAS ORB SHORT — Mar/Abr 2026")
print("=" * 40)
print(f"Total trades:        {total}")
print(f"Ganadores:           {len(ganadores)} ({win_rate:.1f}%)")
print(f"Perdedores:          {len(perdedores)} ({100-win_rate:.1f}%)")
print(f"Por target:          {por_resultado.get('target', 0)}")
print(f"Por stop:            {por_resultado.get('stop', 0)}")
print(f"Por tiempo:          {por_resultado.get('tiempo', 0)}")
print(f"PnL total:           ${pnl_total:,.0f}")
print(f"Ganancia media:      ${ganancia_media:,.0f}")
print(f"Pérdida media:       ${perdida_media:,.0f}")
print(f"Profit factor:       {profit_factor:.2f}")
print(f"Max drawdown:        ${max_drawdown:,.0f}")
print("=" * 40)

# Curva de equity
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, total+1)),
    y=df_trades["equity"],
    mode="lines+markers",
    line=dict(color="royalblue", width=2),
    marker=dict(
        color=["green" if p > 0 else "red" for p in df_trades["pnl"]],
        size=8
    ),
    name="Equity"
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Curva de equity — ORB SHORT",
    xaxis_title="Trade #",
    yaxis_title="PnL acumulado (USD)",
    height=400,
    template="plotly_white"
)
fig.show()

ESTADÍSTICAS ORB SHORT — Mar/Abr 2026
Total trades:        13
Ganadores:           5 (38.5%)
Perdedores:          8 (61.5%)
Por target:          5
Por stop:            8
Por tiempo:          0
PnL total:           $-838
Ganancia media:      $930
Pérdida media:       $-686
Profit factor:       0.85
Max drawdown:        $3,250
